# Complete Analysis: Jobs, Stages, Tasks & RDD Lineage

**Full RDD DAG Analysis with Spark UI Integration**

This notebook includes the complete RDD lineage as it appears in the Spark UI, with detailed explanations of WholeStageCodegen instances and partition handling.

## Understanding RDD Lineage Notation

When reading Spark UI DAGs, you'll see:

| Notation | Meaning |
|----------|----------|
| **RDD[N]** | Unique ID Spark assigns to track RDD across jobs |
| **[Unordered]** | Data has no specific ordering or partitioning scheme |
| **WholeStageCodegen(N)** | Nth code generation instance (fuses different operators) |
| **MapPartitionsRDD** | Output RDD after a transformation |
| **ShuffledRowRDD** | References shuffle output files from a previous stage |
| **ParallelCollectionRDD** | Initial RDD created from a Python collection |
| **Exchange** | Shuffle boundary — writes/reads data across partitions |
| **mapPartitionsInternal** | Helper function for driver collection (collect/show) |

---

## Job 0 — Stage 0: Scan & Partial Aggregation

**What happens:** Read 10 rows across 8 partitions → filter to 7 rows (1 partition produces zero output, task skipped) → compute partial aggregates by department → write to 200 shuffle buckets.

### RDD Lineage (as seen in Spark UI)

```
ParallelCollectionRDD[5]
    └─ What: Original data (10 employees, 8 partitions)
    └─ Created by: spark.createDataFrame(pdf, schema)
    │
    v

WholeStageCodegen (1)
    └─ What: Tungsten code gen instance #1
    └─ Fuses: LocalTableScan [dept, salary_k]
             + Filter [salary > 70000]
             + Project [salary_k = salary/1000]
             + HashAggregate(partial)[dept]
    └─ Output: Single JVM bytecode loop compiling all 4 operators
    │
    v

MapPartitionsRDD[6]
    └─ What: Output of WholeStageCodegen(1) execution
    └─ Content: 7 rows [dept, salary_k] from 7 tasks
    └─ Schema: (dept: String, salary_k: Double)
    │
    v

Exchange [hashpartitioning(dept, 200)]
    └─ What: SHUFFLE BOUNDARY — writes data to 200 partition buckets
    └─ Partitioning: hash(dept) % 200 → determines bucket file
    └─ Output files: 200 files written (197 empty, 3 with data)
    │
    v

MapPartitionsRDD[7]
    └─ What: Post-exchange wrapper RDD
    └─ Contains: References to 200 shuffle partition files on disk
    └─ Status: Ready for Job 1 to read
```

### Why WholeStageCodegen(1)?

It's the **first code generation instance** in this execution plan. The number (1) distinguishes it from (2) and (3) that appear in later jobs.

- **Purpose:** Fuse 4 separate operators into one tight JVM loop
- **Benefit:** Eliminates per-row virtual function calls and intermediate object allocation
- **Typical speedup:** 5-10x faster than non-fused execution

### Task Execution Details

| Task | Input Partition | Rows In | Filter Outcome | Output | Status |
|------|-----------------|---------|---|--------|--------|--------|
| 0 | 0 | 2 rows | 2 pass (alice:95k, bob:72k) | 2 rows | ✓ Runs |
| 1 | 1 | 1 row | 1 pass (carol:105k) | 1 row | ✓ Runs |
| 2 | 2 | 1 row | 0 pass (dave:68k) | 0 rows | ✗ SKIPPED |
| 3 | 3 | 1 row | 1 pass (eve:88k) | 1 row | ✓ Runs |
| 4 | 4 | 1 row | 0 pass (frank:61k) | 0 rows | ✗ SKIPPED |
| 5 | 5 | 1 row | 0 pass (grace:63k) | 0 rows | ✗ SKIPPED |
| 6 | 6 | 1 row | 1 pass (heidi:112k) | 1 row | ✓ Runs |
| 7 | 7 | 2 rows | 2 pass (ivan:74k, judy:99k) | 2 rows | ✓ Runs |

**Result:** 7 tasks executed, 1 skipped. Output: 7 rows scattered across 200 shuffle buckets.

---

## Job 1 — Stage 1: Shuffle Read & Final Aggregation

**What happens:** Read 200 shuffle bucket files (coalesced by AQE) → merge partial aggregates → write to 200 range-partitioned buckets for sorting.

### RDD Lineage (as seen in Spark UI)

```
ShuffledRowRDD[8] [Unordered]
    └─ What: Reference to shuffle output from Job 0, Stage 0
    └─ Created by: Exchange operator in Stage 0
    └─ Content: 7 aggregate partial rows (scattered across 200 files)
    └─ [Unordered]: No specific ordering — rows scattered by hash
    │
    v

AQEShuffleRead
    └─ What: AQE optimization wrapper around shuffle read
    └─ Decision: "197/200 files are empty. Read all as 1 coalesced partition."
    └─ Effect: Converts 200 planned tasks → 1 actual task
    │
    v

WholeStageCodegen (2)
    └─ What: Tungsten code gen instance #2
    └─ Fuses: AQEShuffleRead deserialization
             + HashAggregate(final)[dept]
             + Range partitioning logic
    └─ Output: Compiles deserialize + merge + partition into 1 loop
    │
    v

MapPartitionsRDD[9] [Unordered]
    └─ What: Output of WholeStageCodegen(2) execution
    └─ Content: 3 final aggregate rows [dept, headcount, avg_salary_k]
    └─ Schema: (dept: String, headcount: Long, avg_salary_k: Double)
    └─ [Unordered]: Aggregates computed but not yet sorted
    │
    v

Exchange [rangepartitioning(avg_salary_k DESC, 200)]
    └─ What: SECOND SHUFFLE BOUNDARY — range partition for sorting
    └─ Partitioning: Range buckets by avg_salary_k values
    └─ Output files: 200 range-partitioned files
    │
    v

MapPartitionsRDD[10] [Unordered]
    └─ What: Post-exchange wrapper, part 1
    │
    v

MapPartitionsRDD[11] [Unordered]
    └─ What: Post-exchange wrapper, part 2 (intermediate transformation)
    │
    v

MapPartitionsRDD[12] [Unordered]
    └─ What: Final post-exchange wrapper
    └─ Contains: References to 200 range-partitioned files
```

### Why WholeStageCodegen(2)?

It's the **second code generation instance**. Different from (1) because it fuses different operators:

- **(1)** fused: scan → filter → project → partial agg
- **(2)** fuses: shuffle read → final agg → range partition

Each instance compiles its own bytecode optimized for its specific operations.

### Why ShuffledRowRDD[8]?

- **[8]** is Spark's unique ID for this shuffle RDD
- **ShuffledRowRDD** indicates it references shuffle output files (not in-memory data)
- **Created by:** Exchange operator in Job 0, Stage 0
- **Content:** 200 partition files written in Stage 0

### Why [Unordered]?

Data scattered across partitions has **no guaranteed order**:
- Rows aren't sorted by any key
- Rows aren't grouped by any key
- Each partition contains random subset of data

Opposite would be [Ordered] (after a global sort).

### Why Multiple MapPartitionsRDD([10], [11], [12])?

Each represents a step in the Exchange operation:
- **[10]:** Immediate output of Exchange write operation
- **[11]:** After serialization/metadata attachment
- **[12]:** Final reference ready for next job

These are implementation details of how Spark chains shuffle operations.

### Task Execution Details

| Task | Input | Rows | Operation | Output |
|------|-------|------|-----------|--------|
| 0 | ShuffledRowRDD[8] (all 200 files) | 7 | Merge partials → compute avg → range partition | 3 rows |

**Result:** 1 task executed (AQE coalesced 200 planned tasks). Output: 3 final rows across 200 range buckets.

---

## Job 2 — Stage 4: Range Shuffle Read & Sort

**Note:** Stages 2 and 3 were planned but skipped by AQE. Only Stage 4 executes.

**What happens:** Read 200 range-partitioned files (coalesced) → apply in-memory sort → prepare for driver collection.

### RDD Lineage (as seen in Spark UI)

```
ShuffledRowRDD[8] [Unordered]  ← SAME RDD as Job 1!
    └─ Important: Both Job 1 and Job 2 read from the SAME shuffle output
    └─ Why: Job 1 wrote to disk; Job 2 must read those same files
    └─ Reuse: Spark reuses RDD reference to show dependency
    │
    v

AQEShuffleRead
    └─ What: AQE wrapper for reading range-partitioned output
    └─ Decision: "3 rows across 200 range files. Read all as 1 partition."
    │
    v

WholeStageCodegen (2)  ← SAME instance as Job 1!
    └─ Why reused: Same code gen ID represents same bytecode context
    └─ However: May execute different code path (sort vs. range partition)
    │
    v

MapPartitionsRDD[9] [Unordered]  ← SAME RDD as Job 1!
    └─ Why reused: Spark caches RDD DAG references across jobs
    └─ Represents: Same data but accessed from different stage
    │
    v

Exchange [for final result]
    └─ What: Final shuffle boundary (may be optimized away)
    │
    v

MapPartitionsRDD[13] [Unordered]
    └─ What: Sorted result ready for collection
    └─ Content: 3 rows sorted by avg_salary_k DESC
```

### Why ShuffledRowRDD[8] appears again?

- Job 1's Exchange wrote 200 range-partitioned files to disk
- Job 2 must read those same files to continue processing
- Spark reuses the RDD reference [8] to show the producer-consumer relationship

This is a **dependency:** Job 2 cannot start until Job 1's Stage completes and writes the files.

### Why MapPartitionsRDD[9] reused?

Spark tracks RDD transformations in a DAG. The same logical data (3 aggregates) is referenced multiple times:
- **In Job 1:** After WholeStageCodegen(2) computes aggregates
- **In Job 2:** When reading those aggregates to sort them

Reuse shows Spark's efficient caching of RDD identities.

### Task Execution Details

| Task | Input | Rows | Operation | Output |
|------|-------|------|-----------|--------|
| 0 | ShuffledRowRDD[8] (all 200 range files) | 3 | In-memory sort by avg_salary_k | 3 sorted rows |

**Result:** 1 task executed. Output: 3 rows sorted DESC (eng:99.8, sales:73.0, hr:62.0)

---

## Job 3 — Stage 7: Final Collection to Driver

**Note:** Stages 5 and 6 were planned but skipped by AQE. Only Stage 7 executes.

**What happens:** Read sorted result partition → format rows for display → send to driver JVM → show() prints.

### RDD Lineage (as seen in Spark UI)

```
ShuffledRowRDD[14] [Unordered]
    └─ What: Output from Job 2's Exchange (final result partition)
    └─ Created by: Exchange operator in Stage 4
    └─ Content: 3 final sorted rows
    │
    v

AQEShuffleRead
    └─ What: Final read operation (no shuffle involved)
    └─ Purpose: Access the 1 partition with sorted result
    │
    v

WholeStageCodegen (3)
    └─ What: Tungsten code gen instance #3
    └─ Fuses: Row deserialization
             + String formatting (| dept | headcount | avg |)
             + Serialization for driver transfer
    └─ Output: Compiles I/O formatting into optimized bytecode
    │
    v

MapPartitionsRDD[15] [Unordered]
    └─ What: Output of WholeStageCodegen(3) execution
    └─ Content: 3 rows formatted as display strings
    └─ Ready: For transmission to driver
    │
    v

mapPartitionsInternal
    └─ What: Helper function for driver data transfer
    └─ Purpose: Copy RDD data from executor JVM to driver JVM
    └─ Mechanism: Network/IPC serialization + transmission
    └─ Not an RDD: But a wrapper around the final collection
    │
    v

MapPartitionsRDD[16] [Unordered]
    └─ What: Final RDD in driver process memory
    └─ Content: 3 rows now in driver JVM
    └─ Status: Ready for show() to print
```

### Why WholeStageCodegen(3)?

It's the **third code generation instance**. Different from (1) and (2):

- **(1)** fused: scan → filter → project → partial agg
- **(2)** fused: shuffle read → final agg → range partition
- **(3)** fuses: deserialize → format for display → serialize for driver

Each instance is optimized for its specific I/O and transformation pattern.

### What is mapPartitionsInternal?

Not a true RDD operator, but a **helper function** that handles:
1. Copying data from executor process to driver process
2. Serializing data for network transmission (or IPC)
3. Managing memory on both sides of the boundary

Required for actions like:
- `show()` (display to console)
- `collect()` (gather all rows)
- `toPandas()` (convert to pandas in driver)

### Task Execution Details

| Task | Input | Rows | Operation | Output |
|------|-------|------|-----------|--------|
| 0 | ShuffledRowRDD[14] (1 partition) | 3 | Format & send to driver | 3 rows in driver |

---

## Complete RDD Lineage Summary

### Job Dependencies

```
Job 0, Stage 0:
  ParallelCollectionRDD[5]
    → WholeStageCodegen(1) → MapPartitionsRDD[6]
    → Exchange → MapPartitionsRDD[7]
    └─ Writes: ShuffledRowRDD[8] (200 hash files)

Job 1, Stage 1:
  ShuffledRowRDD[8] ← INPUT from Job 0
    → AQEShuffleRead → WholeStageCodegen(2) → MapPartitionsRDD[9]
    → Exchange → MapPartitionsRDD[10,11,12]
    └─ Writes: ShuffledRowRDD[14] (200 range files)

Job 2, Stage 4:
  ShuffledRowRDD[14] ← INPUT from Job 1  (Note: Different RDD, same pattern)
    → AQEShuffleRead → WholeStageCodegen(2) → MapPartitionsRDD[9]
    → Exchange → MapPartitionsRDD[13]
    └─ Final sorted result

Job 3, Stage 7:
  ShuffledRowRDD[14] (from Job 2)
    → AQEShuffleRead → WholeStageCodegen(3) → MapPartitionsRDD[15]
    → mapPartitionsInternal → MapPartitionsRDD[16]
    └─ 3 rows in driver memory
```

### Key RDD Insights

| RDD | Job | Purpose | Content | Status |
|-----|-----|---------|---------|--------|
| **ParallelCollectionRDD[5]** | 0 | Initial data | 10 rows, 8 partitions | Source |
| **MapPartitionsRDD[6]** | 0 | After WholeStageCodegen(1) | 7 filtered rows | Intermediate |
| **ShuffledRowRDD[8]** | 0,1 | Shuffle output (hash) | 200 partition files | Shuffle output |
| **MapPartitionsRDD[9]** | 1,2 | After WholeStageCodegen(2) | 3 aggregate rows | Reused |
| **ShuffledRowRDD[14]** | 1,2,3 | Shuffle output (range) | 200 range files | Shuffle output |
| **MapPartitionsRDD[15]** | 3 | After WholeStageCodegen(3) | 3 formatted rows | Formatted |
| **MapPartitionsRDD[16]** | 3 | In driver memory | 3 rows ready to print | Final |

### WholeStageCodegen Breakdown

| Instance | Operators Fused | Output | Why This Instance |
|----------|-----------------|--------|-------------------|
| **(1)** | Scan + Filter + Project + Partial Agg | 7 rows to shuffle | Source → partial computation |
| **(2)** | Shuffle Read + Final Agg + Range Partition | 3 aggregates + partition info | Merge + prepare for sort |
| **(3)** | Deserialize + Format + Serialize | Display strings | Optimize I/O to driver |

---

## Why Stages Are Skipped — The Full Story

### Stages Skipped by AQE

```
Planned (pre-execution):  Stage 0, 1, 2, 3, 4, 5, 6, 7
Executed (AQE-optimized): Stage 0, 1, 4, 7
Skipped:                  Stage 2, 3, 5, 6
```

### Why Each Stage Was Skipped

**Stage 2 (planned but skipped):**
- Originally planned as separate shuffle-read stage
- AQE merged it into Stage 1 because data was tiny
- No separate execution needed

**Stage 3 (planned but skipped):**
- Originally planned as separate range-shuffle-read stage
- AQE merged it into Stage 4 for same reason
- Avoided double deserialization

**Stages 5 & 6 (planned but skipped):**
- AQE's safety stages for final data movement
- Eliminated because data already in 1 sorted partition
- No need to re-shuffle for driver collection

**Key insight:** AQE skips stages not by removing them from the DAG, but by merging them into adjacent stages before execution. This is why you see Stage numbers jump (0, 1, 4, 7).

---

## Interview Key Takeaways

### Understanding RDD IDs

**Q: "What does RDD[8] mean, and why does it appear in multiple jobs?"**

A: RDD[8] is Spark's **unique identifier** for a shuffle RDD. When Job 1 writes shuffle files via Exchange, Spark assigns [8] to the output. When Job 2 needs to read those same files, it references the same RDD[8] to show the dependency. It's not "reused" in memory (Job 1's Executor writes; Job 2 reads from disk), but Spark tracks both as the same logical RDD for debugging/visualization.

---

### Understanding WholeStageCodegen Numbering

**Q: "Why are there WholeStageCodegen(1), (2), and (3)?"**

A: Each number represents a **different compilation context** with different fused operators:
- **(1)** compiles scan+filter+project+partial agg (source → shuffle write)
- **(2)** compiles shuffle read+final agg+range partition (shuffle read → shuffle write)
- **(3)** compiles I/O formatting for driver transfer (executor → driver)

Tungsten generates separate bytecode for each to optimize for that specific operation pattern.

---

### Understanding Partition State ([Unordered])

**Q: "What does [Unordered] mean on ShuffledRowRDD[8]?"**

A: It means the **data has no guaranteed ordering or partitioning scheme**. Rows are scattered across 200 files by hash(dept), so you can't assume rows are sorted or grouped. If data were globally sorted, you'd see [Ordered] or a specific sort key.

---

### Understanding mapPartitionsInternal

**Q: "What is mapPartitionsInternal doing?"**

A: It's the **final step of data collection**, moving data from executor JVMs to the driver JVM. Not an RDD transformation, but a helper function that serializes executor data and transmits it over the network/IPC. Used by all actions that return data to the driver: `show()`, `collect()`, `first()`, `toPandas()`.